<a href="https://colab.research.google.com/github/Wambo44-master/pedestrian-safety-dashboard/blob/main/pedestrian_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install streamlit
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import plotly.express as px
import plotly.graph_objects as go

# ============================================
# PAGE CONFIGURATION
# ============================================
st.set_page_config(
    page_title="Kenya Pedestrian Safety Dashboard",
    page_icon="🚶",
    layout="wide"
)

# ============================================
# TITLE AND INTRODUCTION
# ============================================
st.title("🚶‍♂️ Kenya Pedestrian Safety Analysis")
st.markdown("""
**Data-driven insights for road safety policy**
Based on KNBS Road Accident Casualty Data (2018-2021)
""")

# ============================================
# YOUR DATA (from the report)
# ============================================
years = [2018, 2019, 2020, 2021]
casualties = [2312, 2950, 3186, 3667]
fatalities = [882, 1144, 1109, 1248]

# Fatality rates by road user
road_users = ['Pedestrians', 'Pedal Cyclist', 'Motor Cyclist', 'Drivers', 'Pillion Passengers', 'Passengers']
fatality_rates = [45.70, 44.60, 30.92, 22.46, 19.61, 10.22]

# Odds ratios from logistic regression
odds_data = {
    'Road User': ['Pedestrians', 'Motor Cyclists', 'Drivers', 'Passengers'],
    'Odds Ratio (vs Drivers)': [2.89, 1.38, 1.00, 0.46],
    'Interpretation': [
        '2.89x more likely to die',
        '1.38x more likely to die',
        'Reference group',
        '0.46x less likely to die'
    ]
}
odds_df = pd.DataFrame(odds_data)

# ============================================
# TRAIN PREDICTION MODEL
# ============================================
lr_model = LinearRegression()
lr_model.fit([[y] for y in years], casualties)

# ============================================
# SIDEBAR - MAKE PREDICTIONS
# ============================================
st.sidebar.header("🔮 Predict Future Pedestrian Casualties")

prediction_year = st.sidebar.number_input(
    "Enter year:",
    min_value=2022,
    max_value=2030,
    value=2025,
    step=1
)

if st.sidebar.button("Predict", type="primary"):
    pred_casualties = lr_model.predict([[prediction_year]])[0]
    pred_fatalities = pred_casualties * 0.457  # 45.7% fatality rate

    st.sidebar.success(f"### Predictions for {prediction_year}")
    st.sidebar.metric(
        "Predicted Pedestrian Casualties",
        f"{int(pred_casualties):,}"
    )
    st.sidebar.metric(
        "Predicted Pedestrian Fatalities",
        f"{int(pred_fatalities):,}"
    )
    st.sidebar.caption(
        f"Based on trend: +430 casualties/year (R² = 0.972)"
    )

# ============================================
# KEY METRICS (Top Row)
# ============================================
col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric(
        "Avg Annual Pedestrian Casualties",
        "3,029",
        delta="+430/year",
        delta_color="inverse"
    )

with col2:
    st.metric(
        "Pedestrian Fatality Rate",
        "45.7%",
        help="Highest among all road users"
    )

with col3:
    st.metric(
        "Pedestrian Share of Fatalities (2021)",
        "34%",
        help="Down from 38% in 2018"
    )

with col4:
    st.metric(
        "Odds vs Drivers",
        "2.89x",
        help="Pedestrians are nearly 3x more likely to die"
    )

# ============================================
# CHART 1: TRENDS OVER TIME
# ============================================
st.subheader("📈 Pedestrian Casualties & Fatalities (2018-2021)")

fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=years, y=casualties,
    mode='lines+markers',
    name='Casualties',
    line=dict(color='blue', width=3),
    marker=dict(size=10)
))
fig1.add_trace(go.Scatter(
    x=years, y=fatalities,
    mode='lines+markers',
    name='Fatalities',
    line=dict(color='red', width=3),
    marker=dict(size=10)
))

# Add trend line
z = np.polyfit(years, casualties, 1)
p = np.poly1d(z)
fig1.add_trace(go.Scatter(
    x=years, y=p(years),
    mode='lines',
    name='Trend Line',
    line=dict(color='green', dash='dash')
))

fig1.update_layout(
    title="Pedestrian Casualties and Fatalities",
    xaxis_title="Year",
    yaxis_title="Count",
    hovermode='x unified'
)
st.plotly_chart(fig1, use_container_width=True)

# ============================================
# CHART 2: FATALITY RATES BY ROAD USER
# ============================================
st.subheader("⚠️ Fatality Rates by Road User")

fig2 = px.bar(
    x=road_users,
    y=fatality_rates,
    color=road_users,
    title="Fatality Rate When Involved in an Accident (2018-2021)",
    labels={'x': 'Road User', 'y': 'Fatality Rate (%)'},
    text=fatality_rates
)
fig2.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig2.update_layout(showlegend=False, height=500)
st.plotly_chart(fig2, use_container_width=True)

# ============================================
# CHART 3: ODDS RATIOS
# ============================================
st.subheader("📊 Fatality Odds Comparison")

fig3 = px.bar(
    odds_df,
    x='Road User',
    y='Odds Ratio (vs Drivers)',
    color='Road User',
    title="Odds of Fatality Compared to Drivers (Logistic Regression)",
    text='Odds Ratio (vs Drivers)'
)
fig3.update_traces(texttemplate='%{text:.2f}x', textposition='outside')
fig3.add_hline(y=1, line_dash="dash", line_color="red",
               annotation_text="Driver baseline")
fig3.update_layout(height=500)
st.plotly_chart(fig3, use_container_width=True)

# ============================================
# KEY FINDINGS (Expandable Sections)
# ============================================
st.subheader("📋 Key Findings from Analysis")

with st.expander("🔍 Click to view detailed findings"):
    col1, col2 = st.columns(2)

    with col1:
        st.markdown("""
        **Pedestrian Share of Fatalities**
        - 2018: 38.16%
        - 2019: 38.76%
        - 2020: 34.79%
        - 2021: 34.02%

        **Trend Analysis**
        - Annual increase: **+430 casualties/year**
        - Model fit: **R² = 0.972** (very strong)
        - Statistical significance: **p = 0.014**
        """)

    with col2:
        st.markdown("""
        **Logistic Regression Results**
        - Pedestrians: **2.89x** more likely to die than drivers
        - Statistically significant (p < 0.001)
        - Confirms pedestrian vulnerability

        **ARIMA Forecast (2022-2023)**
        - Predicted fatalities: ~1,575-1,577 deaths
        - Continued high numbers without intervention
        """)

# ============================================
# RECOMMENDATIONS
# ============================================
st.subheader("💡 Policy Recommendations")

rec_col1, rec_col2 = st.columns(2)

with rec_col1:
    st.info("""
    **🚧 Infrastructure Improvements**
    - Speed humps in high-risk areas
    - Zebra crossings with proper lighting
    - Pedestrian footbridges at busy intersections
    - Dedicated sidewalks separated from traffic
    """)

with rec_col2:
    st.warning("""
    **📢 Enforcement & Awareness**
    - Campaigns on yielding to pedestrians
    - Stricter penalties for hit-and-run
    - Safe crossing education for pedestrians
    - Improved street lighting at night
    """)

st.success("""
**📊 Data & Planning**
- Regular release of county-level accident data
- Use forecasting models to anticipate resource needs
- Multi-sectoral collaboration (NTSA + County + Urban Planners)
""")

# ============================================
# FOOTER
# ============================================
st.divider()
st.caption("""
**Source:** KNBS Transport Data Portal - Road Accident Casualties by Road User (2018-2021)
**Project:** Data Science Capstone - Pedestrian Safety Analysis
**Note:** Predictions are based on historical trends and assume no major policy changes
""")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 130.6 MB/s eta 0:00:00


2026-04-10 09:37:14.289 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 09:37:14.290 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 09:37:14.439 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-04-10 09:37:14.439 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 09:37:14.441 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 09:37:14.442 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 09:37:14.443 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

DeltaGenerator()